# 04 — PPO-based RLHF Training

**RLHF Pipeline for Compact Open-Source LLMs**

This notebook runs Proximal Policy Optimization (PPO) to align the SFT model using reward model scores. This is the core RLHF step.

## Key Considerations

- **TRL version sensitivity:** This notebook targets `trl==0.9.6`. The PPO API changes between TRL versions.
- **Policy model:** SFT model loaded as base + LoRA adapter + value head (two-step approach)
- **Reward model:** Trained in notebook 03, loaded for scoring generated responses
- **Monitoring:** Mean reward, KL divergence, and response length are logged per step
- **RQ3:** High KL divergence may indicate policy instability or collapse. Response length inflation may indicate reward hacking.

## RLHF Pipeline

```
Prompt -> Policy generates response -> Reward model scores -> PPO update
```

## 4.1 Environment and Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

from src.data_utils import set_seed, load_dataset_from_disk
from src.model_utils import load_tokenizer
from src.reward_train import compute_reward_scores
from src.ppo_train import PPOLogger

set_seed(42)

import trl
print(f"TRL version: {trl.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Configuration
MODEL_NAME       = "Qwen/Qwen2.5-0.5B"
SFT_MODEL_DIR    = PROJECT_ROOT / "outputs" / "models" / "sft_qwen"
REWARD_MODEL_DIR = PROJECT_ROOT / "outputs" / "models" / "reward_model_hh"
PPO_OUTPUT_DIR   = PROJECT_ROOT / "outputs" / "models" / "ppo_qwen"
LOG_DIR          = PROJECT_ROOT / "outputs" / "logs"
FIGURES_DIR      = PROJECT_ROOT / "outputs" / "figures"

for d in [PPO_OUTPUT_DIR, LOG_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# PPO hyperparameters
PPO_STEPS      = 500      # Total PPO steps (reduce for testing)
BATCH_SIZE     = 16
MINI_BATCH     = 4
PPO_EPOCHS     = 4
MAX_NEW_TOKENS = 128
MAX_PROMPT_LEN = 256

## 4.2 Load Policy Model (SFT + Value Head)

We use a two-step loading approach validated in the smoke test:
1. Load base causal LM
2. Attach SFT LoRA adapter via PEFT
3. Wrap with `AutoModelForCausalLMWithValueHead`

In [ ]:
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from peft import PeftModel
from transformers import AutoModelForCausalLM

tokenizer = load_tokenizer(MODEL_NAME)

# Step 1: base model + SFT LoRA adapter
compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype=compute_dtype,
)
base_model = PeftModel.from_pretrained(base_model, str(SFT_MODEL_DIR))
print("SFT LoRA adapter attached")

# Step 2: wrap with value head
policy = AutoModelForCausalLMWithValueHead.from_pretrained(base_model)
print("Value head wrapped")

n_total = sum(p.numel() for p in policy.parameters())
n_trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"Policy params — total: {n_total:,}  trainable: {n_trainable:,}")

## 4.3 Load Reward Model

In [ ]:
from src.model_utils import load_peft_model

rm_cfg = {
    "seed": 42,
    "training": {"fp16": True, "bf16": False},
}

reward_model = load_peft_model(
    MODEL_NAME, REWARD_MODEL_DIR, rm_cfg, is_reward_model=True,
)
reward_model.eval()
rm_tokenizer = load_tokenizer(MODEL_NAME)
print("Reward model loaded and set to eval mode")

## 4.4 Prepare Prompts

In [ ]:
pref_data = load_dataset_from_disk(PROJECT_ROOT / "data" / "processed" / "hh_rlhf" / "train")
prompts = pref_data["prompt"]
print(f"PPO prompts available: {len(prompts)}")

## 4.5 PPO Configuration and Training

In [ ]:
ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=BATCH_SIZE,
    mini_batch_size=MINI_BATCH,
    ppo_epochs=PPO_EPOCHS,
    init_kl_coef=0.2,
    target=6.0,
    gamma=1.0,
    lam=0.95,
    cliprange=0.2,
    cliprange_value=0.2,
    seed=42,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy,
    tokenizer=tokenizer,
)

gen_kwargs = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "min_new_tokens": 16,
    "temperature": 0.7,
    "top_k": 50,
    "top_p": 0.95,
    "do_sample": True,
    "pad_token_id": tokenizer.pad_token_id,
}

print(f"PPO config: batch={BATCH_SIZE}, mini_batch={MINI_BATCH}, ppo_epochs={PPO_EPOCHS}")
print(f"PPO steps: {PPO_STEPS}")

In [ ]:
# Run this cell to start PPO training
ppo_logger = PPOLogger(LOG_DIR / "ppo_training_log.csv")
prompt_idx = 0

for step in range(PPO_STEPS):
    # Batch of prompts
    batch_prompts = [
        prompts[i % len(prompts)]
        for i in range(prompt_idx, prompt_idx + BATCH_SIZE)
    ]
    prompt_idx += BATCH_SIZE

    # Tokenize
    query_tensors = [
        tokenizer.encode(p, return_tensors="pt", truncation=True,
                         max_length=MAX_PROMPT_LEN).squeeze(0)
        for p in batch_prompts
    ]

    # Generate
    full_tensors = ppo_trainer.generate(query_tensors, return_prompt=False, **gen_kwargs)

    # Strip query prefix if needed (TRL version compatibility)
    response_tensors = []
    for i, full in enumerate(full_tensors):
        full = full.squeeze()
        q_len = query_tensors[i].shape[0]
        if full.shape[0] > q_len and torch.equal(full[:q_len], query_tensors[i]):
            response_tensors.append(full[q_len:])
        else:
            response_tensors.append(full)

    # Decode
    response_texts = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]

    # Reward scores
    combined = [f"{p}\n\n{r}" for p, r in zip(batch_prompts, response_texts)]
    rewards = compute_reward_scores(reward_model, rm_tokenizer, combined, max_length=512, batch_size=BATCH_SIZE)
    reward_tensors = [torch.tensor(r, dtype=torch.float32) for r in rewards]

    # PPO step
    stats = ppo_trainer.step(query_tensors, response_tensors, reward_tensors)

    # Log
    mean_reward = sum(rewards) / len(rewards)
    mean_len = sum(len(r) for r in response_texts) / len(response_texts)
    kl = None
    for kl_key in ("ppo/mean_kl", "ppo/kl", "objective/kl"):
        if kl_key in stats:
            kl = stats[kl_key]
            break

    ppo_logger.log(step, {
        "mean_reward": mean_reward,
        "response_length_mean": mean_len,
        "kl_divergence": kl,
    })

    if step % 10 == 0:
        print(f"Step {step}/{PPO_STEPS} — reward={mean_reward:.4f}  len={mean_len:.0f}  kl={kl:.4f if isinstance(kl, (int, float)) else 'N/A'}")

ppo_logger.save()

# Save aligned model
policy.save_pretrained(str(PPO_OUTPUT_DIR))
tokenizer.save_pretrained(str(PPO_OUTPUT_DIR))
print(f"PPO-aligned model saved to {PPO_OUTPUT_DIR}")

## 4.6 PPO Training Curves

These curves are critical for RQ3 analysis:
- **Mean reward** should increase if RLHF is working
- **KL divergence** should remain bounded; high KL indicates policy collapse
- **Response length** inflation may indicate reward hacking / verbosity bias

In [ ]:
log_path = LOG_DIR / "ppo_training_log.csv"

if log_path.exists():
    ppo_df = pd.read_csv(log_path)
    print(f"PPO log: {len(ppo_df)} rows, columns: {list(ppo_df.columns)}")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Mean reward curve
    axes[0].plot(ppo_df["step"], ppo_df["mean_reward"], color="steelblue")
    axes[0].set_xlabel("PPO Step")
    axes[0].set_ylabel("Mean Reward")
    axes[0].set_title("PPO Mean Reward Curve")
    
    # KL divergence curve
    kl_data = ppo_df.dropna(subset=["kl_divergence"])
    if not kl_data.empty:
        axes[1].plot(kl_data["step"], kl_data["kl_divergence"], color="red")
        axes[1].axhline(y=6.0, color="gray", linestyle="--", alpha=0.5, label="Target KL=6.0")
        axes[1].set_xlabel("PPO Step")
        axes[1].set_ylabel("KL Divergence")
        axes[1].set_title("PPO KL Divergence")
        axes[1].legend()
    
    # Response length curve
    axes[2].plot(ppo_df["step"], ppo_df["response_length_mean"], color="green")
    axes[2].set_xlabel("PPO Step")
    axes[2].set_ylabel("Mean Response Length (chars)")
    axes[2].set_title("PPO Response Length Trend")
    
    fig.suptitle("PPO Training Diagnostics", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "ppo_training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("TO BE FILLED AFTER RUNNING PPO TRAINING")

## 4.7 Sample Generations: Before vs After PPO

**TO BE FILLED AFTER RUNNING FULL EXPERIMENTS**

Compare outputs from the SFT model and the PPO-aligned model on the same prompts.

In [ ]:
# This cell compares SFT vs PPO outputs on sample prompts.
# Requires both models to be trained and saved.

eval_prompts = [
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
    "How can I improve my sleep quality?",
]

print("TO BE FILLED AFTER RUNNING FULL PPO TRAINING")
print("Load both SFT and PPO models, generate for the same prompts, compare side by side.")

## Summary

| Output | Location |
|---|---|
| PPO-aligned model | `outputs/models/ppo_qwen/` |
| PPO training log CSV | `outputs/logs/ppo_training_log.csv` |
| PPO training curves | `outputs/figures/ppo_training_curves.png` |

**Key observations for RQ3 (TO BE FILLED AFTER RUNNING):**
- Did mean reward increase over training?
- Was KL divergence stable or did it diverge?
- Did response length inflate (possible reward hacking)?

**Next:** Proceed to `05_evaluation.ipynb` for comprehensive evaluation.